# 5-4: Named Entity Recognition With spaCy

In this tutorial, we'll learn how to use __named entity recognition__, often shortened to __NER__, with `spaCy`. We'll use the same NYT election article dataset from our previous tutorials.

NER identifies text that refers to named things: people, organizations, places, events, and so on. For DH work, NER can help us move from raw text to structured information. For example, we might ask which people appear most often in a news corpus, which places are mentioned over time, or which organizations tend to appear in the same articles.

By the end, you should be able to explain what NER does, why spaCy's NER is a machine-learning approach, and how to turn NER output into tabular data for humanities analysis.

## Learning Objectives

1. Explain what named entity recognition tries to identify
2. Explain how spaCy's NER component uses a trained machine-learning model
3. Load a spaCy English model and inspect its pipeline components
4. Extract entities from one text and from a dataframe of texts
5. Store entity text, labels, article metadata, and character offsets in a dataframe
6. Count common entity labels and common entities
7. Use NER output to ask article-level and corpus-level questions
8. Compare machine-learning NER with a simple rule-based search
9. Interpret NER mistakes carefully


## What Is Named Entity Recognition?

__Named entity recognition__ is a method for finding and labeling named _things_ in text.

For example, in the sentence:

> Kamala Harris campaigned in Wisconsin before Election Day.

An NER model might identify:

| entity | label |
| --- | --- |
| `Kamala Harris` | `PERSON` |
| `Wisconsin` | `GPE` |
| `Election Day` | `DATE` |

NER is useful because it turns text into structured information. Once entities are in a dataframe, we can count them, group them, plot them over time, compare them by article section, or use them as the basis for a network analysis.

But NER is not the same thing as understanding a text. It identifies spans and labels. It does not know why those people or places matter. And its tags or labels are sometimes incorrect.

## spaCy NER: Machine Learning

The NER component in spaCy's English pipeline is a __trained model__. That means it learned from examples where human annotators marked entities in text.

During training, the model sees many examples like:

| text span | human label |
| --- | --- |
| `Joseph R. Biden Jr.` | `PERSON` |
| `the White House` | `ORG` |
| `Pennsylvania` | `GPE` |

The model learns patterns from those examples. When we give it a new article, it predicts which text are entities and which labels they should receive.

This is different from a simple dictionary lookup. A dictionary lookup only finds words or names we've already listed. A machine-learning NER model can often identify new names it has never seen before because it's learned patterns around capitalization, word order, context, and grammar.

That flexibility is useful, but it also creates errors. The model can miss entities, split names in odd ways, or assign the wrong label.

### Important Interpretive Caution

NER models are not neutral readers. They are trained on particular datasets with particular annotation rules.

That matters for DH projects. A model may work well on contemporary news but less well on older texts, poetry, OCR-transcribed newspapers, multilingual texts, or writing with unusual spelling and punctuation. 

It may also reflect the categories of its training data. For instance, spaCy's `GPE` label means geopolitical entity, such as a country, city, or state. That is useful, but it is also a specific way of organizing place references. It's not the only way to think of places––what about general regions (i.e., The West, the Midwest, the Deep South, etc.)? This `GPE` label doesn't account for those more nuanced versions of places.

As you work through this notebook, keep asking:

- What kinds of entities does the model recognize?
- Which entities does it miss?
- Which labels seem confusing or wrong?
- What would a human reader notice that the model cannot?
- How might the genre of news writing shape the results?


## The Election Article Dataset

We'll use `../data/election2020_articles.csv`, the same dataset from the sentiment-analysis notebook.

Each row represents one New York Times article collected around the 2020 presidential election. The dataset includes article metadata such as headline, publication date, section, news desk, abstract, and lead paragraph.

For this tutorial, we'll run NER on a short text field made from each article's headline plus lead paragraph (just as we did in the sentiment analysis tutorial).

## Setup

We'll use:

- `pandas` for tabular data
- `matplotlib` and `seaborn` for plots
- `spaCy` for named entity recognition
- `displaCy`, spaCy's visualization tool, for viewing entities in one article

spaCy's NER system requires a trained language pipeline. We'll use `en_core_web_sm`, the small English pipeline.

A spaCy pipeline is a sequence of processing steps that run when we call `nlp(text)`. First, spaCy tokenizes the text, turning the string into a `Doc` object made of tokens. Then different pipeline components add annotations to that `Doc`. Depending on the pipeline, those components might assign part-of-speech tags, dependency parses, lemmas, sentence boundaries, or named entities.

A trained pipeline also includes model data, or learned weights, that some components use to make predictions. In our case, the important component is `ner`, the named entity recognizer. The small English pipeline is fast and convenient for class, but "small" means it is a compact general-purpose model. It will not be perfect, and larger or domain-specific pipelines may behave differently.

For more detail, see spaCy's documentation on [language processing pipelines](https://spacy.io/usage/processing-pipelines) and [English trained pipelines](https://spacy.io/models/en).

In [ ]:
%matplotlib inline

from itertools import combinations
import string

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import spacy
from spacy import displacy
from spacy.cli import download

In [ ]:
model_name = "en_core_web_sm"

try:
    nlp = spacy.load(model_name)
except OSError:
    download(model_name)
    nlp = spacy.load(model_name)

nlp.pipe_names


The object `nlp` is a spaCy pipeline. When text passes through it, spaCy tokenizes the text and runs several components.

The component we care about most today is `ner`, which identifies named entities. Notice that not every part of the pipeline is machine learning in the same way. Tokenization uses language-specific rules. The `ner` component, however, is a trained machine-learning model.


In [ ]:
ner = nlp.get_pipe("ner")

label_guide = pd.DataFrame({
    "label": list(ner.labels),
})
label_guide["description"] = label_guide["label"].apply(spacy.explain)

label_guide

## Step 1: Load The Data

Let's read the local CSV file and preview the columns that will matter for this lesson.


In [ ]:
articles = pd.read_csv("../data/election2020_articles.csv")

articles.shape

In [ ]:
preview_columns = [
    "headline.main",
    "lead_paragraph",
    "pub_date",
    "section_name",
    "news_desk",
]

articles[preview_columns].head()

Let's check for missing values in the text columns we might use.


In [ ]:
articles[["headline.main", "lead_paragraph", "abstract"]].isna().sum()

## Step 2: Build A Text Column

We'll create a new column called `analysis_text` by combining each article's headline and lead paragraph. If the lead paragraph is missing, we'll use the abstract as a backup.

This is a choice. If we could run NER on full articles, we would find many more entities. If we ran NER on headlines only, we would find fewer entities and probably miss important context. Remember: the model needs context in order to classify named entities effectively.

In [ ]:
headline_text = articles["headline.main"].fillna("")
lead_text = articles["lead_paragraph"].fillna(articles["abstract"].fillna(""))

articles["analysis_text"] = headline_text + " " + lead_text
articles["analysis_text"] = articles["analysis_text"].str.replace(r"\s+", " ", regex=True).str.strip()

articles = articles[articles["analysis_text"].str.len() > 20].copy()
articles["pub_date"] = pd.to_datetime(articles["pub_date"])

articles[["headline.main", "analysis_text"]].head()

Here is one example text. Before running any model, it is always a good idea to read at least a few examples by hand.


In [ ]:
sample_index = articles.index[5]
sample_text = articles.loc[sample_index, "analysis_text"]

sample_text

## Step 3: Run NER On One Example

Now we'll pass one article through the spaCy pipeline.

The result is a `Doc` object. This object contains tokens, sentences, part-of-speech information, and named entities.


In [ ]:
sample_doc = nlp(sample_text)

type(sample_doc)

Named entities are stored in `doc.ents`. Let's inspect them as simple tuples first.


In [ ]:
entity_tuples = []

for ent in sample_doc.ents:
    entity_tuples.append((ent.text, ent.label_))

entity_tuples


The loop above moves through each entity span in `sample_doc.ents`. For each entity, `ent.text` gives us the exact text span and `ent.label_` gives us spaCy's string label, such as `PERSON`, `ORG`, or `GPE`.

That tuple view is compact. A dataframe would be easier to read. So let's write a helper function that converts a spaCy `Doc` into a dataframe of entities.

In [ ]:
def entities_to_dataframe(doc):
    entity_rows = []

    for ent in doc.ents:
        entity_rows.append({
            "entity": ent.text,
            "label": ent.label_,
            "description": spacy.explain(ent.label_),
            # to keep the original string indices
            "start_char": ent.start_char,
            "end_char": ent.end_char,
        })

    return pd.DataFrame(entity_rows)


entities_to_dataframe(sample_doc)

The `start_char` and `end_char` columns tell us where the entity begins and ends in the original string. Those offsets are useful when we want to connect the entity back to its exact context.

spaCy also includes an entity visualizer called `displaCy`. It highlights entity spans and labels directly in the text. Pretty neat!

In [ ]:
displacy.render(sample_doc, style="ent", jupyter=True)

### Challenge: Inspect One Article

Choose a different article index and rerun the NER code on that article.

Try asking:

1. Which entities did spaCy identify correctly?
2. Which entities did it miss?
3. Did it assign any labels that seem wrong?
4. Would the headline alone have given enough context?


In [ ]:
# Try changing this number
new_sample_index = articles.index[25]
new_sample_text = articles.loc[new_sample_index, "analysis_text"]
new_sample_doc = nlp(new_sample_text)

print(new_sample_text)
entities_to_dataframe(new_sample_doc)

## Step 4: Process The Full Dataset

Now that we understand the output for one article, we can process all the article texts.

For many texts, spaCy recommends `nlp.pipe()`. This processes texts in batches and is faster than running `nlp(text)` one row at a time.

We'll save one row per entity. Each entity row will include:

- the article index
- the entity text
- the entity label
- the article headline
- the publication date
- the section and news desk
- the entity's character offsets in the article text

In [ ]:
article_metadata = articles[[
    "headline.main",
    "pub_date",
    "section_name",
    "news_desk",
]].to_dict("index")

entity_rows = []
texts = articles["analysis_text"].tolist()
article_indices = articles.index.tolist()

for article_index, doc in zip(article_indices, nlp.pipe(texts, batch_size=50)):
    metadata = article_metadata[article_index]

    for ent in doc.ents:
        entity_rows.append({
            "article_index": article_index,
            "entity": ent.text,
            "label": ent.label_,
            "description": spacy.explain(ent.label_),
            "start_char": ent.start_char,
            "end_char": ent.end_char,
            "headline": metadata["headline.main"],
            "pub_date": metadata["pub_date"],
            "section_name": metadata["section_name"],
            "news_desk": metadata["news_desk"],
        })

entities = pd.DataFrame(entity_rows)
entities["pub_date"] = pd.to_datetime(entities["pub_date"])

entities.shape


In [ ]:
entities.head(10)


This dataframe is the bridge between text and analysis. We've converted unstructured article text into a structured table of entity mentions. Neat!

Each row is an entity mention, not a unique real-world person or place. That distinction matters. For example, `Trump`, `Donald Trump`, and `Donald J. Trump` may all refer to the same person, but they appear as different text spans.

Are there any strange discrepancies in the above examples? It's worth taking a close look.

## Step 5: Count Entity Labels

Let's start by asking which kinds of entities spaCy found most often.


In [ ]:
entity_label_counts = entities["label"].value_counts()
entity_label_counts


In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(
    x=entity_label_counts.head(12).values,
    y=entity_label_counts.head(12).index,
    color="steelblue",
)
plt.title("Most Common Entity Labels")
plt.xlabel("Number of entity mentions")
plt.ylabel("Entity label")
plt.tight_layout()

Not all labels are equally useful for every project. In a news corpus, `PERSON`, `ORG`, `GPE`, and `DATE` are often especially useful. But a different research question might care more about laws, works of art, money, or events.

Let's create a small guide for the labels that actually appear in our dataset.


In [ ]:
present_label_guide = pd.DataFrame({
    "label": entity_label_counts.index,
    "count": entity_label_counts.values,
})
present_label_guide["description"] = present_label_guide["label"].apply(spacy.explain)

present_label_guide

## Step 6: Clean Entity Text For Counting

Counting exact entity spans can be messy. The same person might appear as `Biden`, `Joe Biden`, `Joseph R. Biden Jr.`, or `Joseph R. Biden Jr.'s`.

So let's do a light cleanup that removes extra whitespace and strips punctuation from the beginning and end of entity strings. This doesn't solve every normalization problem, but it helps a little.

The function below takes one entity string at a time. First, `entity_text.split()` breaks the string into pieces wherever there is whitespace. Then `" ".join(...)` puts those pieces back together with single spaces, which removes extra spaces, tabs, or line breaks. Next, `.strip(string.punctuation)` removes punctuation from the beginning and end of the string, such as quotation marks, periods, commas, or possessive apostrophes. Finally, the function returns the cleaned version.

After defining the function, we use `.apply()` to run it on every value in the `entity` column. Then we drop very short cleaned strings, since one-character entities are usually not useful for our counting here.

In [ ]:
def clean_entity_text(entity_text):
    entity_text = entity_text.lower()
    cleaned = " ".join(entity_text.split())
    cleaned = cleaned.strip(string.punctuation)
    return cleaned


entities["entity_clean"] = entities["entity"].apply(clean_entity_text)
entities = entities[entities["entity_clean"].str.len() > 1].copy()

entities[["entity", "entity_clean", "label"]].head(10)

Now let's look at the most common entities in a few major labels.

The helper function below lets us reuse the same counting logic for different entity labels. It filters the entity table to one label, groups the rows by cleaned entity text, counts how many times each cleaned entity appears, sorts the counts from highest to lowest, and returns the top `n` results.


In [ ]:
def top_entities(label, n=15):
    label_entities = entities[entities["label"] == label]

    return (
        label_entities
        .groupby("entity_clean")
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(n)
    )


top_entities("PERSON")


In [ ]:
top_entities("ORG")

In [ ]:
top_entities("GPE")

### Reading Model Output Carefully

Look closely at those lists. Some entries probably make sense. Others may look strange.

For example, a model might label a surname as a person in one sentence and as an organization in another. It might treat `WASHINGTON` as a place, an organization, or a dateline depending on context. It might also include possessive forms or partial names.

These are not just annoying cleanup problems. They are interpretive evidence about how the model sees the text.


### Challenge: Investigate One Entity

Choose one common entity from the `PERSON`, `ORG`, or `GPE` tables.

Then ask:

1. Does this entity appear in multiple forms?
2. Does spaCy always assign the same label to that entity?
3. Are there obvious false positives?
4. Would you need to normalize these mentions before making a research claim?


## Step 7: Compare Entity Labels By Article Section

Because we kept article metadata, we can compare entity counts by section.

Let's ask: which sections contain the most entity mentions in this dataset?


In [ ]:
section_entity_counts = (
    entities
    .groupby("section_name")
    .size()
    .reset_index(name="entity_mentions")
    .sort_values("entity_mentions", ascending=False)
)

section_entity_counts.head(10)


In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(
    data=section_entity_counts.head(10),
    x="entity_mentions",
    y="section_name",
    color="steelblue",
)
plt.title("Entity Mentions By Article Section")
plt.xlabel("Number of entity mentions")
plt.ylabel("Section")
plt.tight_layout()


Raw counts can be misleading because some sections have more articles than others. Let's also calculate average entity mentions per article by section.

The code below first counts how many entity rows belong to each article. Then it maps those counts back onto the main `articles` dataframe as a new `entity_count` column. Articles with no detected entities get `0` with `.fillna(0)`.

After that, we group the article dataframe by `section_name`. For each section, `.agg()` calculates three summary values: how many articles are in that section, the average number of entity mentions per article, and the total number of entity mentions. Sorting by `mean_entity_count` helps us see which sections tend to have more entity-dense blurbs, even if those sections have fewer total articles.

In [ ]:
article_entity_counts = entities.groupby("article_index").size().rename("entity_count")
articles["entity_count"] = articles.index.map(article_entity_counts).fillna(0).astype(int)

section_summary = (
    articles
    .groupby("section_name")
    .agg(
        articles=("analysis_text", "size"),
        mean_entity_count=("entity_count", "mean"),
        total_entity_count=("entity_count", "sum"),
    )
    .sort_values("mean_entity_count", ascending=False)
)

section_summary.head(10)


Average counts help us ask a slightly different question: not just which sections dominate the dataset, but which sections tend to have entity-dense article blurbs.


In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(
    data=section_summary.reset_index().head(10),
    x="mean_entity_count",
    y="section_name",
    color="steelblue",
)
plt.title("Average Entity Mentions Per Article By Section")
plt.xlabel("Average entity mentions per article")
plt.ylabel("Section")
plt.tight_layout()


## Step 8: Create Article-Level Entity Features

NER can also create new article-level features. For each article, we can count how many `PERSON`, `ORG`, and `GPE` mentions appeared.

This turns the model output into columns we can analyze with ordinary pandas methods.

The code below starts from the entity-level dataframe and reshapes it into an article-level table. `groupby(["article_index", "label"]).size()` counts how many times each entity label appears in each article. `.unstack(fill_value=0)` turns the labels into columns, so each row is one article and each column is a label count.

Then the `for` loop copies a few useful label counts back into the `articles` dataframe. For example, `PERSON` becomes `person_count`, `ORG` becomes `org_count`, and so on. The `if` statement protects us in case a label does not appear in the current dataset; in that case, we create the column and fill it with zeros.

In [ ]:
label_count_table = (
    entities
    .groupby(["article_index", "label"])
    .size()
    .unstack(fill_value=0)
)

for label in ["PERSON", "ORG", "GPE", "DATE"]:
    if label in label_count_table.columns:
        articles[f"{label.lower()}_count"] = (
            articles.index.map(label_count_table[label]).fillna(0).astype(int)
        )
    else:
        articles[f"{label.lower()}_count"] = 0

articles[[
    "headline.main",
    "entity_count",
    "person_count",
    "org_count",
    "gpe_count",
    "date_count",
]].head()


Which articles have the most entity mentions?


In [ ]:
articles.sort_values("entity_count", ascending=False)[[
    "headline.main",
    "section_name",
    "entity_count",
    "person_count",
    "org_count",
    "gpe_count",
    "analysis_text",
]].head(10)


A histogram shows the distribution of entity counts across article blurbs.


In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(
    data=articles,
    x="entity_count",
    bins=25,
    color="steelblue",
)
plt.title("Entity Mentions Per Article")
plt.xlabel("Number of entity mentions")
plt.ylabel("Number of articles")
plt.tight_layout()


## Step 9: Entity Labels Over Time

Because the dataset includes publication dates, we can also look at entity labels over time.

Here we'll count four labels by week: `PERSON`, `ORG`, `GPE`, and `DATE`.


In [ ]:
labels_to_plot = ["PERSON", "ORG", "GPE", "DATE"]

weekly_label_counts = (
    entities[entities["label"].isin(labels_to_plot)]
    .set_index("pub_date")
    .groupby("label")
    .resample("W")
    .size()
    .unstack(0)
    .fillna(0)
)

weekly_label_counts

In [ ]:
ax = weekly_label_counts.plot(figsize=(10, 5), marker="o")
plt.title("Entity Labels By Week")
plt.xlabel("Publication week")
plt.ylabel("Number of entity mentions")
plt.tight_layout()

This plot doesn't explain why entity mentions rise or fall. It only tells us where to look more closely.

A DH workflow often moves back and forth like this: use a computational method to find a pattern, then return to the texts and historical context to interpret it.

## Step 10: Find Articles That Mention An Entity

Let's write a small helper function that finds articles mentioning a particular entity.

This is a nice way to move from distant reading back toward close reading.

The function below takes two arguments: `entity_name`, the name or partial name we want to search for, and `max_rows`, the maximum number of articles to return.

Inside the function, we search the cleaned entity text rather than the full article text. The `.str.lower()` calls make the search case-insensitive, and `regex=False` tells pandas to treat the search string as ordinary text rather than a regular expression. After finding matching entity mentions, the function gets the unique article IDs, keeps only the first few with `.head(max_rows)`, and returns those article rows from the main `articles` dataframe.

This is a substring search, so searching for `Trump` can match `Donald Trump`, `Trump's`, or another cleaned entity string that contains those letters.

In [ ]:
def show_articles_with_entity(entity_name, max_rows=5):
    # Find all entities that match the given entity name (case-insensitive)
    matches = entities[
        entities["entity_clean"].str.lower().str.contains(
            entity_name.lower(),
            regex=False,
        )
    ]

    # Get the unique article IDs from the matches
    article_ids = matches["article_index"].drop_duplicates().head(max_rows)

    return articles.loc[article_ids, [
        "headline.main",
        "pub_date",
        "section_name",
        "analysis_text",
    ]]


show_articles_with_entity("Kamala Harris")

Try changing the entity name above. For example, you might search for `Biden`, `Florida`, `Supreme Court`, or another entity from the frequency tables.


In [ ]:
show_articles_with_entity("Supreme Court")

## Step 11: Entity Co-Occurrence

NER can also support simple co-occurrence analysis.

For example, we can ask which `PERSON` entities appear in the same article blurbs.

The code below starts by keeping only rows where spaCy assigned the `PERSON` label. Then it groups those person mentions by article and turns each article's names into a sorted set. The `set()` removes duplicate mentions within the same article, and `sorted()` makes the output consistent.

Next, we use `combinations(people, 2)` to create every two-person pair that appears within the same article. Each pair becomes one row in `pair_rows`, along with the article index where the co-occurrence happened.

Finally, we turn those rows into a dataframe and count how often each pair appears across article blurbs. These are co-occurrences, not proven relationships. They only mean that two names appeared in the same article text field.

In [ ]:
person_entities = entities[entities["label"] == "PERSON"].copy()

people_by_article = (
    person_entities
    .groupby("article_index")["entity_clean"]
    .apply(lambda names: sorted(set(names)))
)

pair_rows = []

for article_index, people in people_by_article.items():
    if len(people) > 1:
        for person_1, person_2 in combinations(people, 2):
            pair_rows.append({
                "person_1": person_1,
                "person_2": person_2,
                "article_index": article_index,
            })

person_pairs = pd.DataFrame(pair_rows)

person_pair_counts = (
    person_pairs
    .groupby(["person_1", "person_2"])
    .size()
    .reset_index(name="article_count")
    .sort_values("article_count", ascending=False)
)

person_pair_counts.head(20)

This co-occurrence table is a starting point, not a conclusion. Some pairs are meaningful. Others may be artifacts of common names, partial names, or repeated campaign boilerplate.

If you wanted to turn this into a network analysis, you would need to make careful decisions about entity normalization. Is `Biden` the same as `Joseph R. Biden Jr.`? Is `Trump` the same as `Donald J. Trump`? Usually, yes. But Python will not know that unless we tell it.


## Step 12: Compare NER To A Rule-Based Search

To make the machine-learning part clearer, let's compare spaCy NER to a simple rule-based search.

A rule-based search can answer questions like: does this article contain the exact string `Biden`?

NER asks a different question: which spans does the trained model predict are named entities, and what labels does it assign them?

Both approaches can be useful. They just do different things.

The code below compares those approaches for a short list of names. For each name, the rule-based method searches `analysis_text` for the exact string, ignoring capitalization. The NER method searches only entity rows that spaCy labeled as `PERSON`, then checks whether the cleaned entity text contains the same name.

The two count columns measure articles, not total mentions. `rule_based_article_count` counts how many article blurbs contain the string. `ner_person_article_count` counts how many unique articles contain a matching `PERSON` entity. If the numbers differ, that difference can help us see what the model is adding, missing, or labeling differently.


In [ ]:
comparison_names = ["Trump", "Biden", "Harris", "Pence"]
comparison_rows = []

for name in comparison_names:
    rule_based_articles = articles[
        articles["analysis_text"].str.contains(name, case=False, regex=False)
    ]

    ner_person_articles = entities[
        (entities["label"] == "PERSON")
        & (entities["entity_clean"].str.contains(name, case=False, regex=False))
    ]

    comparison_rows.append({
        "name": name,
        "rule_based_article_count": rule_based_articles.shape[0],
        "ner_person_article_count": ner_person_articles["article_index"].nunique(),
    })

rule_vs_ner = pd.DataFrame(comparison_rows)
rule_vs_ner

These counts may differ for several reasons:

- The rule-based search can find a name anywhere, even if spaCy does not label it as `PERSON`.
- spaCy may identify a longer name, such as `Joseph R. Biden Jr.`, that contains the search string `Biden`.
- spaCy may label the same string differently depending on context.
- A rule-based search cannot identify names we forgot to search for.

This is the tradeoff. Rule-based methods are transparent and targeted. Machine-learning methods are more flexible, but also less predictable.


### A Small Test With New Sentences

Let's give the model a few short sentences that are not from the dataset. This is not a formal evaluation. It's just a way to see how the trained NER component behaves on new text.

In [ ]:
test_texts = [
    "Stacey Abrams spoke in Atlanta after voters waited in long lines.",
    "A campaign group sued Facebook in federal court on Monday.",
    "The town of Truth or Consequences appeared in a voting-rights story.",
    "This sentence may throw off the model: Matthews is a real town."
]

for text in test_texts:
    doc = nlp(text)
    print(text)
    print([(ent.text, ent.label_) for ent in doc.ents])
    print()

Notice that the model is making predictions. It's not checking a perfect database of all people, places, and organizations. That is why it can generalize, and also why it can make surprising mistakes.

## Optional: Save The Entity Table

If you wanted to use the extracted entities in another notebook, you could save them as a CSV file.

The line below is commented out so we do not create extra files by accident. Uncomment it if you want to save the entity table.


In [ ]:
# entities.to_csv("../data/election2020_entities_spacy.csv", index=False)
